# Stage 4: Summary Injection with Gemma-3-12B-IT

This notebook generates two-field JSON summaries for each chunk using `google/gemma-3-12b-it` with 4-bit NF4 quantization.

**Requirements:**
- Kaggle GPU T4 (16 GB VRAM)
- `stage3_chunks.parquet` mounted from Kaggle Dataset
- Hugging Face token with access to gated Gemma models

**Outputs:**
- `stage4_enriched.parquet` — enriched chunks with summaries
- `summary_cache.jsonl` — resumable cache

**Runtime:** ~17 GPU-hours total (split across multiple sessions)

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q transformers==4.46.0 torch==2.4.1 accelerate==1.0.1 bitsandbytes==0.44.1 pandas pyarrow

In [ ]:
import os
import json
import re
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Any

import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
    print(f"CUDA memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 2. Configuration

In [ ]:
# Paths - adjust these based on your Kaggle Dataset mount
INPUT_PATH = Path("/kaggle/input/datasets/boinhbo/glrag-stage3/stage3_chunks.parquet")
OUTPUT_DIR = Path("/kaggle/working")
OUTPUT_PATH = OUTPUT_DIR / "stage4_enriched.parquet"
CACHE_PATH = OUTPUT_DIR / "summary_cache.jsonl"

# Model configuration
MODEL_NAME = "google/gemma-3-12b-it"
MAX_INPUT_CHARS = 3000
MAX_NEW_TOKENS = 256
BATCH_SIZE = 2  # Conservative for 16GB VRAM
TEMPERATURE = 0.1
REPETITION_PENALTY = 1.05

# Hugging Face token - set this in Kaggle secrets or environment
HF_TOKEN = user_secrets.get_secret("HF_TOKEN") or user_secrets.get_secret("HUGGINGFACE_HUB_TOKEN")
if not HF_TOKEN:
    print("WARNING: No HF_TOKEN found. Set it in Kaggle Secrets as HF_TOKEN")
    print("Go to: Add-ons > Secrets > Add a new secret")
else:
    print("✓ HF_TOKEN found")

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input: {INPUT_PATH}")
print(f"Output: {OUTPUT_PATH}")
print(f"Cache: {CACHE_PATH}")

## 3. Prompt Templates

In [ ]:
SYSTEM_PROMPT = """Bạn là chuyên gia tóm tắt văn bản pháp luật Việt Nam."""

USER_PROMPT_TEMPLATE = """Hãy tạo HAI tóm tắt cho điều luật dưới đây:

[ĐIỀU LUẬT]
{chunk_text_truncated}

[YÊU CẦU]
1) SUM_SHORT: 1 câu duy nhất (≤30 từ) nêu CHỦ ĐỀ chính của điều.
2) SUM_KEY: 3-5 gạch đầu dòng nêu các Ý PHÁP LÝ then chốt
   (đối tượng áp dụng, nghĩa vụ, quyền, điều kiện, chế tài).

Trả về JSON: {{"short": "...", "key": ["...", "..."]}}"""

# Regex for JSON extraction
JSON_EXTRACT_RE = re.compile(r'\{[^{}]*"short"\s*:\s*"[^"]*"(?:,\s*"key"\s*:\s*\[[^\]]*\])?[^{}]*\}', re.DOTALL)
JSON_EXTRACT_RE_FALLBACK = re.compile(r'\{.*\}', re.DOTALL)

## 4. Helper Functions

In [ ]:
def load_cache(cache_path: Path) -> Dict[str, Dict]:
    """Load existing cache, return dict mapping chunk_id to summary dict."""
    if not cache_path.exists():
        return {}
    
    cache = {}
    with cache_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                chunk_id = record.get("chunk_id", "")
                if chunk_id:
                    cache[chunk_id] = record
            except json.JSONDecodeError:
                continue
    return cache


def append_to_cache(cache_path: Path, chunk_id: str, short: str, key: List[str]) -> None:
    """Append a single cache entry to the JSONL file."""
    with cache_path.open("a", encoding="utf-8") as f:
        f.write(json.dumps({"chunk_id": chunk_id, "short": short, "key": key}, ensure_ascii=False) + "\n")


def extract_json_from_response(text: str) -> Optional[Dict[str, Any]]:
    """Extract and parse JSON from model response."""
    # Try to find JSON object with short and key fields
    match = JSON_EXTRACT_RE.search(text)
    if match:
        try:
            result = json.loads(match.group())
            if "short" in result and "key" in result:
                return result
        except json.JSONDecodeError:
            pass
    
    # Fallback: try to find any JSON object
    match = JSON_EXTRACT_RE_FALLBACK.search(text)
    if match:
        try:
            result = json.loads(match.group())
            if "short" in result and "key" in result:
                return result
        except json.JSONDecodeError:
            pass
    
    return None


def build_enriched_text(short: str, key: List[str], chunk_text: str) -> str:
    """Build enriched text from summary and chunk body."""
    key_bullets = "\n".join([f"• {k}" for k in key]) if key else ""
    
    parts = []
    if short:
        parts.append(f"[TÓM TẮT] {short}")
    if key_bullets:
        parts.append(f"[Ý CHÍNH]\n{key_bullets}")
    parts.append(f"[NỘI DUNG ĐẦY ĐỦ]\n{chunk_text}")
    
    return "\n\n".join(parts)

## 5. Load Data

In [ ]:
# Load Stage 3 chunks
print(f"Loading Stage 3 chunks from {INPUT_PATH}")
chunks_df = pd.read_parquet(INPUT_PATH)
print(f"Stage 3 chunks loaded: {len(chunks_df):,}")
print(f"Columns: {chunks_df.columns.tolist()}")
print(f"\nSample row:")
print(chunks_df.iloc[0].to_dict())

# Load existing cache
print(f"\nLoading existing cache from {CACHE_PATH}")
cache = load_cache(CACHE_PATH)
print(f"Cache entries loaded: {len(cache):,}")

# Filter out already-cached chunks
chunks_to_process = chunks_df[~chunks_df["chunk_id"].isin(cache.keys())]
print(f"Chunks to process: {len(chunks_to_process):,}")
print(f"Chunks already cached: {len(chunks_df) - len(chunks_to_process):,}")

## 6. Load Model

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    trust_remote_code=True,
)
print("✓ Tokenizer loaded")

# Configure 4-bit NF4 quantization
print("\nConfiguring 4-bit NF4 quantization...")
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# Load model
print(f"Loading model: {MODEL_NAME}")
print("This may take several minutes...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",
)
model.eval()
print("✓ Model loaded successfully")

# Check memory usage
if torch.cuda.is_available():
    print(f"\nGPU memory allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")
    print(f"GPU memory reserved: {torch.cuda.memory_reserved(0) / 1e9:.2f} GB")

## 7. Generate Summaries

In [ ]:
def generate_summary(chunk_text: str) -> Tuple[str, List[str]]:
    """Generate summary for a single chunk. Returns (short, key_list)."""
    # Truncate input
    truncated = chunk_text[:MAX_INPUT_CHARS] if len(chunk_text) > MAX_INPUT_CHARS else chunk_text
    
    # Build messages for chat template
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT_TEMPLATE.format(chunk_text_truncated=truncated)},
    ]
    
    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # Tokenize
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        padding=True,
        truncation=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=TEMPERATURE,
            repetition_penalty=REPETITION_PENALTY,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
    
    # Decode response
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # Extract JSON
    parsed = extract_json_from_response(response)
    if parsed:
        short = parsed.get("short", "")
        key = parsed.get("key", [])
        if isinstance(key, str):
            key = [key]
        return short, key
    
    # Fallback: return empty values
    return "", []

In [ ]:
# Process chunks
if len(chunks_to_process) > 0:
    print(f"Processing {len(chunks_to_process):,} chunks...")
    print(f"Estimated time: {len(chunks_to_process) * 2.5 / 3600:.1f} hours at ~2.5s/chunk")
    print("\nProgress will be saved to cache after each chunk.")
    print("You can safely interrupt and resume later.\n")
    
    failed_count = 0
    
    for idx, row in tqdm(chunks_to_process.iterrows(), total=len(chunks_to_process), desc="Generating summaries"):
        chunk_id = row["chunk_id"]
        chunk_text = row["chunk_text"]
        
        # Generate summary
        try:
            short, key = generate_summary(chunk_text)
            
            # Save to cache
            append_to_cache(CACHE_PATH, chunk_id, short, key)
            cache[chunk_id] = {"chunk_id": chunk_id, "short": short, "key": key}
            
            if not short and not key:
                failed_count += 1
                
        except Exception as e:
            print(f"\nError processing chunk {chunk_id}: {e}")
            # Save empty values to cache to avoid re-processing
            append_to_cache(CACHE_PATH, chunk_id, "", [])
            cache[chunk_id] = {"chunk_id": chunk_id, "short": "", "key": []}
            failed_count += 1
    
    print(f"\n✓ Processing complete")
    print(f"Failed/empty summaries: {failed_count} ({100*failed_count/len(chunks_to_process):.1f}%)")
else:
    print("All chunks already cached. Skipping generation.")

## 8. Build Enriched Parquet

In [ ]:
print("Building enriched parquet from cache...")

results = []
for idx, row in tqdm(chunks_df.iterrows(), total=len(chunks_df), desc="Building enriched records"):
    chunk_id = row["chunk_id"]
    chunk_text = row["chunk_text"]
    
    # Get summary from cache
    cached_record = cache.get(chunk_id, {})
    short = cached_record.get("short", "")
    key = cached_record.get("key", [])
    
    # Build enriched text
    enriched_text = build_enriched_text(short, key, chunk_text)
    
    # Create result record
    result = row.to_dict()
    result["short"] = short
    result["key"] = key
    result["enriched_text"] = enriched_text
    results.append(result)

# Create output DataFrame
output_df = pd.DataFrame(results)

# Ensure key column is stored as JSON string for Parquet compatibility
if "key" in output_df.columns:
    output_df["key"] = output_df["key"].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)

print(f"\nOutput DataFrame shape: {output_df.shape}")
print(f"Columns: {output_df.columns.tolist()}")

## 9. Save Output

In [ ]:
# Write output
print(f"Writing output to {OUTPUT_PATH}")
output_df.to_parquet(OUTPUT_PATH, index=False)
print(f"✓ Wrote {len(output_df):,} enriched chunks to {OUTPUT_PATH}")

# Summary statistics
non_empty_short = output_df["short"].apply(lambda x: bool(x and str(x).strip())).sum()
non_empty_key = output_df["key"].apply(lambda x: bool(x and str(x).strip() and str(x) != "[]")).sum()
print(f"\nNon-empty short summaries: {non_empty_short:,} ({100*non_empty_short/len(output_df):.1f}%)")
print(f"Non-empty key points: {non_empty_key:,} ({100*non_empty_key/len(output_df):.1f}%)")

# Display sample enriched chunk
print("\n" + "="*80)
print("SAMPLE ENRICHED CHUNK")
print("="*80)
sample_idx = output_df[output_df["short"].str.len() > 0].index[0] if (output_df["short"].str.len() > 0).any() else 0
print(f"\nChunk ID: {output_df.loc[sample_idx, 'chunk_id']}")
print(f"\nEnriched Text:\n{output_df.loc[sample_idx, 'enriched_text'][:1000]}...")

## 10. Upload Instructions

After this notebook completes:

1. Download `stage4_enriched.parquet` and `summary_cache.jsonl` from `/kaggle/working`
2. Create a new Kaggle Dataset named `glrag-stage4`
3. Upload both files to the dataset
4. Use in Notebook 03 by mounting the dataset

**Note:** The cache file allows you to resume if the notebook times out. Simply re-run from the beginning and it will skip already-processed chunks.